# Regression Trees and Ensembles — Solution

**Dataset:** Car fuel consumption in litres per 100 km.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance

## 1. Compute a split

Left targets are 10,12,14; right targets are 30,34. Replace the gap with `np.sum((left - left.mean()) ** 2)`. Compare this split after 14 with the supplied alternative after 12. Which has smaller SSE? Why does each leaf predict its mean?

In [ ]:
left = np.array([10, 12, 14])
right = np.array([30, 34])
sse_left = np.sum((left - left.mean()) ** 2)
sse_right = np.sum((right - right.mean()) ** 2)
print('Predictions:', left.mean(), right.mean())
print('Total split SSE:', sse_left + sse_right)
# Compare splitting after 12 instead of after 14.
alternative_left = np.array([10, 12])
alternative_right = np.array([14, 30, 34])
alternative_sse = (np.sum((alternative_left - alternative_left.mean()) ** 2)
                   + np.sum((alternative_right - alternative_right.mean()) ** 2))
print('Alternative split SSE:', alternative_sse)

**Answer:** The split after 14 has the smaller SSE: 16 compared with 226.

- **Split after 14:** the left leaf (10, 12, 14) has mean 12 and SSE 4 + 0 + 4 = 8. The right leaf (30, 34) has mean 32 and SSE 4 + 4 = 8. Total: 16.
- **Split after 12:** the left leaf (10, 12) has mean 11 and SSE 1 + 1 = 2. The right leaf (14, 30, 34) has mean 26 and SSE 144 + 16 + 64 = 224. Total: 226.
- The value 14 belongs with 10 and 12, not with 30 and 34, so the split after 14 is chosen.
- Each leaf predicts its mean because the mean is the single value with the smallest sum of squared errors for that leaf.

## 2. Prepare car data

The preparation is provided. What does `combined` measure?

In [ ]:
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv')
cars = cars.dropna(subset=['engine_size', 'cylinders', 'combined'])
print(cars[['engine_size', 'cylinders', 'combined']].head())
X = cars[['engine_size', 'cylinders']]
y = cars['combined']  # litres per 100 km
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

**Answer:** `combined` is the combined city/highway fuel consumption in litres per 100 km.

- Lower values mean less fuel for the same distance.
- The first rows show values from 7.4 to 11.2 L/100 km.
- As in module 01, 560 cars are used for training and 141 for testing.

## 3. Fit and read a small tree

Replace the gap with `DecisionTreeRegressor(max_depth=2, random_state=42)`. Follow the rules for engine size 2 and 4 cylinders. The `value` in a leaf is its predicted consumption.

In [ ]:
tree = DecisionTreeRegressor(max_depth=2, random_state=42)
tree.fit(X_train, y_train)
plt.figure(figsize=(10, 5))
plot_tree(tree, feature_names=list(X.columns), filled=True, rounded=True, precision=2)
plt.show()
new_car = pd.DataFrame({'engine_size': [2.0], 'cylinders': [4]})
print('Prediction:', tree.predict(new_car))

**Answer:** The car with 2.0 L engine size and 4 cylinders is predicted to use 8.95 L/100 km.

- Root: is engine size ≤ 2.6? Yes (2.0), so go left.
- Next: is engine size ≤ 1.9? No (2.0), so go right.
- This leaf contains 224 training cars with a mean consumption of 8.95 L/100 km. `predict` returns the same value (8.955).
- The four leaves predict 7.71 (engine ≤ 1.9 L, 41 cars), 8.95 (1.9–2.6 L, 224 cars), 11.73 (2.6–3.65 L, 178 cars) and 14.34 (above 3.65 L, 117 cars).
- All splits use engine size; the depth-2 tree never uses cylinders. Cylinders add little once engine size is known.

## 4. See what pruning changes

Two settings are provided as a demonstration. Compare the number of leaves and training errors. Which tree is easier to describe? We use CV in step 6 for the fixed candidate comparison.

In [ ]:
for alpha in [0.0, 0.05]:
    pruned = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    pruned.fit(X_train, y_train)
    print('alpha:', alpha, 'leaves:', pruned.get_n_leaves(),
          'training RMSE:', round(np.sqrt(mean_squared_error(
              y_train, pruned.predict(X_train))), 2))

**Answer:** The pruned tree (alpha = 0.05) is much easier to describe: 8 leaves instead of 37.

- The training RMSE rises only a little, from 1.43 to 1.52 L/100 km.
- So 29 of the 37 leaves reduce the training error by only 0.09 L/100 km. They are likely to describe random details of the training cars.
- A higher training error does not mean worse predictions for new cars; only validation can show that. In step 6, the pruned tree reaches a CV RMSE of 1.61.

## 5. Train three ensembles

The ensemble code is provided. Explain the difference between bagging/forest and boosting. What does the forest OOB R² describe?

In [ ]:
bagging = BaggingRegressor(n_estimators=50, random_state=42)
forest = RandomForestRegressor(n_estimators=100, max_features=1,
                               oob_score=True, random_state=42)
boosting = GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                     learning_rate=0.1, random_state=42)
bagging.fit(X_train, y_train)
forest.fit(X_train, y_train)
boosting.fit(X_train, y_train)
print('Bagging:', bagging.predict(new_car))
print('Forest:', forest.predict(new_car))
print('Boosting:', boosting.predict(new_car))
print('Forest OOB R²:', round(forest.oob_score_, 3))

**Answer:**

- **Bagging** fits 50 trees, each on a bootstrap sample of the 560 training cars, and averages their predictions.
- **Random forest** does the same with 100 trees, but each split may consider only one randomly chosen input (`max_features=1`). This makes the trees more different from each other, so averaging helps more.
- **Boosting** fits 100 small trees (depth 2) one after another. Each new tree corrects the remaining errors of the previous ones, scaled down by the learning rate 0.1.
- For the 2.0 L, 4-cylinder car, all three predict about 8.9 L/100 km (8.91, 8.92, 8.93). This is close to the single tree's 8.95.
- **OOB R² = 0.69:** each car is predicted only by the trees that did not see it in their bootstrap sample. On these out-of-bag predictions, the forest explains 69% of the variation in consumption. This works like a built-in validation without a separate CV.

## 6. Compare fixed candidates using CV

This small comparison is fully supplied. Which candidate has the lowest CV RMSE? The best CV candidate is then evaluated on the test set.

In [ ]:
models = {
    'Linear': LinearRegression(),
    'Pruned tree': DecisionTreeRegressor(ccp_alpha=0.05, random_state=42),
    'Bagging': bagging,
    'Forest': forest,
    'Boosting': boosting,
}
cv_rmse = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5,
                             scoring='neg_mean_squared_error')
    cv_rmse[name] = np.sqrt(-scores).mean()
print(pd.Series(cv_rmse, name='CV RMSE').round(2))
best_name = min(cv_rmse, key=cv_rmse.get)
best_model = models[best_name]
best_model.fit(X_train, y_train)
print('Chosen model:', best_name)
print('Final test RMSE:', round(np.sqrt(mean_squared_error(
    y_test, best_model.predict(X_test))), 2))

**Answer:** The random forest has the lowest CV RMSE: 1.55 L/100 km.

- Ranking by CV RMSE: forest 1.55, bagging 1.56, boosting 1.57, pruned tree 1.61, linear regression 1.73.
- The three ensembles are within 0.02 L/100 km of each other, so they perform practically the same. All are clearly better than the linear model.
- The forest is chosen because of its CV result, without looking at test rows.
- Final test RMSE of the forest: 1.51 L/100 km. The linear model in module 01 had a test RMSE of 1.85 on the same test cars.
- The test RMSE (1.51) differs a little from the CV RMSE (1.55). It is a single estimate from 141 cars.

## 7. Interpret feature importance

Provided code shuffles one forest input at a time on training data. Which input causes a larger loss in R²? This illustration uses training rows; training importance can be optimistic and does not establish causality.

In [ ]:
importance = permutation_importance(forest, X_train, y_train,
                                    n_repeats=5, random_state=42)
print(pd.Series(importance.importances_mean,
                index=X.columns, name='Decrease in R²').round(3))

**Answer:** Engine size causes the much larger loss in R².

- Shuffling engine size lowers training R² by 0.839; shuffling cylinders lowers it by 0.248.
- The forest relies mainly on engine size, which agrees with the small tree in step 3, where every split used engine size.
- Engine size and cylinders are correlated: large engines usually have more cylinders. Correlated inputs can share importance, so the cylinders value may partly reflect engine size.
- The importance is computed on training rows and can be optimistic.
- Importance is predictive evidence, not a causal claim: it shows what the model uses, not what causes consumption.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))